# **Bioinformatics Project - Computational Drug Discovery [Part 3] Descriptor Calculation and Dataset Preparation**

Chanin Nantasenamat

[*'Data Professor' YouTube channel*](http://youtube.com/dataprofessor)

In this Jupyter notebook, we will be building a real-life **data science project** that you can include in your **data science portfolio**. Particularly, we will be building a machine learning model using the ChEMBL bioactivity data.

In **Part 3**, we will be calculating molecular descriptors that are essentially quantitative description of the compounds in the dataset. Finally, we will be preparing this into a dataset for subsequent model building in Part 4.

---

## **Download PaDEL-Descriptor**

In [75]:
!pip install padelpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 52.7 MB/s eta 0:00:00


In [99]:
from padelpy import padeldescriptor

padeldescriptor(
    mol_dir='molecule.smi',
    d_file='descriptors_output.csv',
    fingerprints=True,
    removesalt=True
)

## **Load bioactivity data**

Download the curated ChEMBL bioactivity data that has been pre-processed from Parts 1 and 2 of this Bioinformatics Project series. Here we will be using the **bioactivity_data_3class_pIC50.csv** file that essentially contain the pIC50 values that we will be using for building a regression model.

In [76]:
import pandas as pd

In [77]:
df3 = pd.read_csv('bioactivity_data_2class.csv')

In [78]:
df3

,molecule_chembl_id,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors,pIC50,bioactivity_class
0,CHEMBL3928693,CC(=O)N[C@@H](CC(=O)O)C(=O)N1CCC[C@H]1C(=O)N[C...,801.804,-3.7467,11.0,12.0,6.939302,active
1,CHEMBL459546,COC(=O)[C@@]12OC[C@]34[C@H]([C@@H](O)[C@@H]1O)...,520.531,0.5170,3.0,11.0,7.534766,active
2,CHEMBL1197091,O=C(CC1NCCCC1O)Cn1cnc2cc(Br)c(Cl)cc2c1=O,414.687,1.8845,2.0,5.0,7.651695,active
3,CHEMBL1197091,O=C(CC1NCCCC1O)Cn1cnc2cc(Br)c(Cl)cc2c1=O,414.687,1.8845,2.0,5.0,7.429457,active


In [79]:
selection = ['canonical_smiles','molecule_chembl_id']
df3_selection = df3[selection]
df3_selection.to_csv('molecule.smi', sep='\t', index=False, header=False)

In [80]:
! cat molecule.smi | head -5

CC(=O)N[C@@H](CC(=O)O)C(=O)N1CCC[C@H]1C(=O)N[C@@H](CCC(=O)O)C(=O)N[C@H](C(=O)NCC(=O)N[C@@H](CCC(=O)O)C(=O)N[C@@H](CC(C)C)C(=O)O)[C@@H](C)O	CHEMBL3928693
COC(=O)[C@@]12OC[C@]34[C@H]([C@@H](O)[C@@H]1O)[C@@]1(C)CC(=O)C(O)=C(C)[C@@H]1C[C@H]3OC(=O)[C@H](OC(=O)C=C(C)C)[C@@H]24	CHEMBL459546
O=C(CC1NCCCC1O)Cn1cnc2cc(Br)c(Cl)cc2c1=O	CHEMBL1197091
O=C(CC1NCCCC1O)Cn1cnc2cc(Br)c(Cl)cc2c1=O	CHEMBL1197091


In [81]:
! cat molecule.smi | wc -l

4


## **Calculate fingerprint descriptors**


In [89]:
%%writefile padel.sh
java -Xms512m -Xmx512m -jar PaDEL-Descriptor/PaDEL-Descriptor.jar -removesalt -retained3d-descriptors -file descriptors_output.csv -dir ./ -fingerprints

Writing padel.sh


### **Calculate PaDEL descriptors**

In [100]:
! cat padel.sh

java -Xms512m -Xmx512m -jar PaDEL-Descriptor/PaDEL-Descriptor.jar -removesalt -retained3d-descriptors -file descriptors_output.csv -dir ./ -fingerprints


In [101]:
! bash padel.sh

Error: Unable to access jarfile PaDEL-Descriptor/PaDEL-Descriptor.jar


In [102]:
! ls -l

total 36
-rw-r--r-- 1 root root   773 Sep  1 18:19 bioactivity_data_2class.csv
-rw-r--r-- 1 root root 18459 Sep  1 20:18 descriptors_output.csv
-rw-r--r-- 1 root root   395 Sep  1 20:05 molecule.smi
-rw-r--r-- 1 root root   153 Sep  1 20:10 padel.sh
drwxr-xr-x 1 root root  4096 Aug 24 13:21 sample_data


## **Preparing the X and Y Data Matrices**

### **X data matrix**

In [103]:
df3_X = pd.read_csv('descriptors_output.csv')

In [104]:
df3_X

,Name,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,CHEMBL3928693,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL459546,1,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL1197091,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL1197091,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [105]:
df3_X = df3_X.drop(columns=['Name'])
df3_X

,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,PubchemFP9,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,1,1,1,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


## **Y variable**

### **Convert IC50 to pIC50**

In [106]:
df3_Y = df3['pIC50']
df3_Y

,pIC50
0,6.939302
1,7.534766
2,7.651695
3,7.429457


## **Combining X and Y variable**

In [107]:
dataset3 = pd.concat([df3_X,df3_Y], axis=1)
dataset3

,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,PubchemFP7,PubchemFP8,PubchemFP9,...,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880,pIC50
0,1,1,1,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,6.939302
1,1,1,1,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,7.534766
2,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,7.651695
3,1,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,7.429457


In [108]:
dataset3.to_csv('NRF2_06_bioactivity_data_3class_pIC50_pubchem_fp.csv', index=False)

# **Let's download the CSV file to your local computer for the Part 3B (Model Building).**